# FASE 1: BUSINESS UNDERSTANDING (Analisis Kebutuhan Bisnis)

### Latar Belakang dan Masalah
Pendidikan merupakan pilar utama pembangunan sumber daya manusia. Dalam lingkungan akademis, tingkat kegagalan siswa dapat diminimalisir jika terdapat sistem peringatan dini (early warning system) yang dapat mengidentifikasi siswa yang berisiko mengalami kesulitan akademis. Dataset ini (`student-por.csv`) merepresentasikan performa siswa dalam mata pelajaran bahasa Portugis di dua sekolah menengah.

### Tujuan Proyek
1. **Unsupervised Learning (K-Means Clustering)**: Mengelompokkan siswa berdasarkan profil akademis dan kebiasaan belajar mereka guna memahami karakteristik kelompok siswa berprestasi tinggi vs berisiko.
2. **Supervised Learning (Klasifikasi)**: Memprediksi status kelulusan siswa secara akurat (`Pass` vs `Fail`) menggunakan algoritma *Logistic Regression* dan *Gaussian Naïve Bayes* dengan menghindari kebocoran data (*data leakage*).
3. **Deployment**: Menyediakan parameter model dalam format JSON agar dapat diintegrasikan ke dalam dashboard client-side untuk inferensi waktu nyata (real-time).

### Kriteria Keberhasilan (Success Metrics)
- Model klasifikasi harus mencapai akurasi > 75% pada data pengujian.
- Evaluasi model menggunakan metrik lengkap: Akurasi, Presisi, Recall, F1-Score, serta Confusion Matrix.


# FASE 2: DATA UNDERSTANDING (Pemahaman Data)

Dalam fase ini, kita akan memuat dataset `student-por.csv`, mengidentifikasi dimensi data, tipe data dari setiap variabel, memeriksa nilai yang hilang (missing values), mengecek data duplikat, dan mengeksplorasi statistik deskriptif awal serta korelasi antar variabel.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import json


In [ ]:
df = pd.read_csv("student-por.csv")
print(df.head())


In [ ]:
print("Jumlah Record :", df.shape[0])
print("Jumlah Variabel :", df.shape[1])
print("\nNama Variabel:")
print(df.columns.tolist())


In [ ]:
print(df.dtypes)


In [ ]:
print(df.info())


In [ ]:
print(df.isnull().sum())


In [ ]:
print(df.duplicated().sum())


In [ ]:
print(df.describe())


In [ ]:
selected_features = [
    'studytime',
    'failures',
    'G1',
    'G2',
    'G3'
]
X_cluster = df[selected_features]
print(X_cluster.head())


In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(X_cluster.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Korelasi Fitur Akademis')
plt.show()


In [ ]:
scaler_cluster = StandardScaler()
X_scaled_cluster = scaler_cluster.fit_transform(X_cluster)


# FASE 3: DATA PREPARATION (Persiapan Data)

Fase persiapan data sangat krusial sebelum melakukan pemodelan untuk menjamin kualitas data input.
Langkah-langkah persiapan data yang dilakukan:
1. **Target Variable Definition**: Membuat target biner `Pass` di mana siswa dinyatakan lulus jika nilai akhir `G3 >= 10` (bernilai 1) dan tidak lulus jika `G3 < 10` (bernilai 0).
2. **Data Leakage Prevention**: Menghapus fitur nilai UTS/UAS (`G1`, `G2`, dan `G3`) serta kolom target `Pass` dari matriks fitur `X` agar prediksi model berorientasi pada masa depan (early prediction) sebelum ujian dilaksanakan.
3. **Categorical Encoding**: Mengubah variabel kategorikal teks menjadi numerik menggunakan `LabelEncoder`.
4. **Data Splitting**: Memisahkan data menjadi 80% pelatihan (*train set*) dan 20% pengujian (*test set*).
5. **Feature Scaling**: Melakukan standardisasi fitur numerik menggunakan `StandardScaler` agar memiliki rata-rata 0 dan variansi 1.


In [ ]:
# 1. Membuat variabel target biner: Pass (1 jika G3 >= 10, else 0)
df['Pass'] = (df['G3'] >= 10).astype(int)

# 2. Mencegah Data Leakage dengan menghapus G1, G2, G3, dan Pass dari fitur
X_supervised = df.drop(columns=['G1', 'G2', 'G3', 'Pass'])
y_supervised = df['Pass']

# 3. Encoding fitur kategorikal menggunakan LabelEncoder
categorical_cols = X_supervised.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_supervised.select_dtypes(exclude=['object']).columns.tolist()

X_encoded = X_supervised.copy()
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_supervised[col])
    label_encoders[col] = le

# 4. Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_supervised, test_size=0.2, random_state=42, stratify=y_supervised
)

# 5. Standardisasi fitur numerik
scaler_supervised = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler_supervised.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler_supervised.transform(X_test[numerical_cols])

print("Dimensi data training:", X_train_scaled.shape)
print("Dimensi data testing:", X_test_scaled.shape)


# FASE 4: MODELING (Pemodelan)

Pada fase ini, dua pendekatan pemodelan akan diterapkan:
1. **Unsupervised Learning**: K-Means Clustering untuk segmentasi kelompok siswa. Kita menentukan jumlah cluster (K) optimal menggunakan **Elbow Method**.
2. **Supervised Learning**: 
   - **Logistic Regression**: Model linier klasifikasi biner.
   - **Gaussian Naïve Bayes**: Model probabilitas berbasis teorema Bayes dengan asumsi distribusi normal pada fitur.


In [ ]:
# Menentukan jumlah cluster optimal dengan Elbow Method
wcss = []
k_range = range(1, 11)
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled_cluster)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, wcss, marker='o', linestyle='-', color='#06b6d4', linewidth=2, markersize=6)
plt.title('Gambar 4.4 Grafik Elbow Method (K-Means Clustering)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=12)
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)
kmeans.fit(X_scaled_cluster)
df['Cluster'] = kmeans.labels_


In [ ]:
# Melatih model Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)

# Melatih model Gaussian Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
print("Pelatihan model Supervised selesai!")


# FASE 5: EVALUATION (Evaluasi)

Fase evaluasi membandingkan hasil pemodelan clustering dan klasifikasi:
- Clustering dinilai berdasarkan Silhouette Score dan analisis rata-rata fitur per cluster.
- Klasifikasi dievaluasi menggunakan Akurasi, Presisi, Recall, F1-Score, dan Confusion Matrix untuk memilih model terbaik bagi prediksi masa depan.


In [ ]:
# Silhouette Score untuk Clustering
score = silhouette_score(X_scaled_cluster, kmeans.labels_)
print("Silhouette Score (KMeans):", score)

print("\n===== JUMLAH ANGGOTA CLUSTER =====")
print(df['Cluster'].value_counts())

print("\n===== RATA-RATA FITUR PER CLUSTER =====")
cluster_summary = df.groupby('Cluster')[selected_features].mean()
print(cluster_summary)


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled_cluster)

plt.figure(figsize=(10,6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df['Cluster'], palette='Set1')
plt.title('Visualisasi Cluster K-Means dengan PCA')
plt.xlabel('Komponen Utama 1')
plt.ylabel('Komponen Utama 2')
plt.show()


In [ ]:
# Prediksi Logistic Regression
y_pred_lr = lr_model.predict(X_test_scaled)
print("=== LOGISTIC REGRESSION EVALUATION ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred_lr):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

# Prediksi Naive Bayes
y_pred_nb = nb_model.predict(X_test_scaled)
print("\n=== GAUSSIAN NAIVE BAYES EVALUATION ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Precision : {precision_score(y_test, y_pred_nb):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred_nb):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred_nb):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))


# FASE 6: DEPLOYMENT (Penyebaran)

Untuk memfasilitasi integrasi model dengan dashboard web di sisi klien tanpa memerlukan server backend Python yang berjalan secara terus-menerus, parameter matematis dari model yang dilatih akan diekspor dalam format JSON.


In [ ]:
# Ekstraksi parameter Logistic Regression
lr_params = {
    'coefficients': lr_model.coef_[0].tolist(),
    'intercept': float(lr_model.intercept_[0])
}

# Ekstraksi parameter Gaussian Naive Bayes
nb_params = {
    'class_prior': nb_model.class_prior_.tolist(),
    'theta': nb_model.theta_.tolist(),      # mean untuk setiap kelas dan fitur
    'var': nb_model.var_.tolist()          # variance untuk setiap kelas dan fitur
}

# Menggabungkan semua metadata deployment
deployment_payload = {
    'feature_names': X_encoded.columns.tolist(),
    'categorical_features': categorical_cols,
    'numerical_features': numerical_cols,
    'scaling_mean': scaler_supervised.mean_.tolist() if hasattr(scaler_supervised, 'mean_') else [],
    'scaling_scale': scaler_supervised.scale_.tolist() if hasattr(scaler_supervised, 'scale_') else [],
    'logistic_regression': lr_params,
    'naive_bayes': nb_params
}

# Print sebagai objek JSON yang bisa langsung di-copy
json_payload = json.dumps(deployment_payload, indent=2)
print("=== JSON PAYLOAD FOR CLIENT-SIDE DASHBOARD INFERENCE ===")
print(json_payload)


In [ ]:
df.to_csv("hasil_clustering.csv", index=False)
print("Hasil clustering disimpan ke hasil_clustering.csv!")
